In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# supprime la notation scientifique
pd.set_option("display.float_format", "{:.2f}".format)
np.set_printoptions(suppress=True)  


%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Chargement des données non traitées dans POSTGRES

In [3]:
# Chargement du fichier CSV
df = pd.read_csv(r"C:\Users\HP\Downloads\data_churn.txt", sep=",")

print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")
df.head()

Nombre de lignes : 528883
Nombre de colonnes : 34


,CUSTOMER_NO,ACCOUNT_NO,NATIONALITY,RESIDENCE,MARITAL_STATUS,CUST_OPENING_DATE,DATE_OF_BIRTH,NATURE_CLIENT,BRANCH,SCORE_KYC,...,PRODUCT_LINE,PRODUCT,ACCOUNTNATURE,STARTDATE,MATURITYDATE,AMOUNT,FIXEDRATE,PRODUCT_STATUS,PARTYCLASS,LOB
0,C318650,A0365322,TN,TN,M,20040930.00,1969.00,PPH,BR114,LR,...,LENDING,RT.RT.CRD.IMMOBILERS.527,Crédit acquisition logement TEGF6,1251227.00,1290627.00,10954600.00,4.50,CURRENT,Retail,4
1,C318648,A0373555,TN,TN,M,20040930.00,1960.00,PPH,BR114,LR,...,DEPOSITS,BANK.CAT.NEG.SIM,DEPOTS A TERME,20260102.00,NaN,NaN,NaN,UNAUTH,Retail,4
2,C318650,A0348290,TN,TN,M,20040930.00,1969.00,PPH,BR114,LR,...,LENDING,RT.RT.CRD.IMMOBILERS.548,Crédit rénovation,1251227.00,1380527.00,113593077.00,4.50,CURRENT,Retail,4
3,C318650,A0257995,TN,TN,M,20040930.00,1969.00,PPH,BR114,LR,...,ACCOUNTS,BANK.CUR.ACCT.ALL.TOURS.CARTE,Compte Allocation Touristique TND,NaN,NaN,0.00,NaN,NaN,Retail,4
4,C318648,A0312991,TN,TN,M,20040930.00,1960.00,PPH,BR114,LR,...,NaN,NaN,DEPOTS A TERME,NaN,NaN,NaN,NaN,NaN,Retail,4


In [4]:
# Pour afficher toutes les colonnes (largeur infinie)
pd.set_option('display.max_columns', None)


# Pour éviter que le texte soit coupé dans les cellules
pd.set_option('display.max_colwidth', None)

In [5]:
df.head()

,CUSTOMER_NO,ACCOUNT_NO,NATIONALITY,RESIDENCE,MARITAL_STATUS,CUST_OPENING_DATE,DATE_OF_BIRTH,NATURE_CLIENT,BRANCH,SCORE_KYC,COMPLETED_FILE,LAST_REVIEW_DATE,NEXT__REVIEW_DATE,ACCOUNT_STATUS,ACCT_OPENING_DATE,ACCOUNT_CATEGORY,ACCOUNT_TYPE_DESC,CURRENCY,ACCT_CLOSE_DATE,CLOSURE_REASON,ACCT_BALANCE,INDUSTRY,SALARY,PRODUCT_GROUP,PRODUCT_LINE,PRODUCT,ACCOUNTNATURE,STARTDATE,MATURITYDATE,AMOUNT,FIXEDRATE,PRODUCT_STATUS,PARTYCLASS,LOB
0,C318650,A0365322,TN,TN,M,20040930.00,1969.00,PPH,BR114,LR,YES,20250905.00,20290905.00,Closed,20190827.00,3023.00,Crédit acquisition logement TEGF6,TND,20260128.00,NaN,-10714.35,9000,2725.74,RT.CRD.IMMOBILERS,LENDING,RT.RT.CRD.IMMOBILERS.527,Crédit acquisition logement TEGF6,1251227.00,1290627.00,10954600.00,4.50,CURRENT,Retail,4
1,C318648,A0373555,TN,TN,M,20040930.00,1960.00,PPH,BR114,LR,YES,20250905.00,20290905.00,Closed,20260105.00,3611.00,DEPOTS A TERME,TND,20250630.00,NaN,0.00,9000,3300.54,BANK.PLACEMENT.NEG,DEPOSITS,BANK.CAT.NEG.SIM,DEPOTS A TERME,20260102.00,NaN,NaN,NaN,UNAUTH,Retail,4
2,C318650,A0348290,TN,TN,M,20040930.00,1969.00,PPH,BR114,LR,YES,20250905.00,20290905.00,Closed,20230612.00,3017.00,Crédit rénovation,TND,20260128.00,NaN,-113033.10,9000,2725.74,RT.CRD.IMMOBILERS,LENDING,RT.RT.CRD.IMMOBILERS.548,Crédit rénovation,1251227.00,1380527.00,113593077.00,4.50,CURRENT,Retail,4
3,C318650,A0257995,TN,TN,M,20040930.00,1969.00,PPH,BR114,LR,YES,20250905.00,20290905.00,Closed,20220527.00,1011.00,Compte Allocation Touristique TND,TND,20260128.00,NaN,0.00,9000,2725.74,BANK.GRP.CUR.ACCT,ACCOUNTS,BANK.CUR.ACCT.ALL.TOURS.CARTE,Compte Allocation Touristique TND,NaN,NaN,0.00,NaN,NaN,Retail,4
4,C318648,A0312991,TN,TN,M,20040930.00,1960.00,PPH,BR114,LR,YES,20250905.00,20290905.00,Closed,20250702.00,3611.00,DEPOTS A TERME,TND,20250630.00,NaN,0.00,9000,3300.54,NaN,NaN,NaN,DEPOTS A TERME,NaN,NaN,NaN,NaN,NaN,Retail,4


In [8]:
from sqlalchemy import create_engine

In [11]:
from sqlalchemy import create_engine
import time

# 1. Paramètres de connexion à ta base
utilisateur = 'postgres'        
mot_de_passe = 'postgres' 
hote = 'localhost'                # Ton PC local
port = '5432'                     # Port par défaut
nom_bdd = 'PIProject'             # Le nom de ta base

# Création du moteur de connexion
chaine_connexion = f'postgresql+psycopg2://{utilisateur}:{mot_de_passe}@{hote}:{port}/{nom_bdd}'
engine = create_engine(chaine_connexion)

print("Connexion à PIProject établie. Début du transfert...")
debut = time.time()

# 2. Envoi du DataFrame (que l'on suppose s'appeler 'df') vers Postgres
df.to_sql(
    name='data_churn',    # Le nom de la table qui sera créée automatiquement dans ta base
    con=engine,                # Le moteur de connexion
    if_exists='replace',       # 'replace' pour écraser et recréer la table si tu relances la cellule
    index=False,               # On ne stocke pas l'index technique de Pandas (0, 1, 2...)
    chunksize=10000            # Taille des paquets envoyés
)

fin = time.time()
print(f"Transfert terminé avec succès en {round(fin - debut, 2)} secondes !")

Connexion à PIProject établie. Début du transfert...
Transfert terminé avec succès en 172.06 secondes !


 ### Traitement des lignes dupliquées (drop_duplicates)

In [6]:
before = len(df)
df = df.drop_duplicates()
print(f"Doublons supprimés : {before - len(df):,}  ({before:,} → {len(df):,} lignes)")

Doublons supprimés : 38,640  (528,883 → 490,243 lignes)


### Traitement des valeurs manquantes qualitatives (Suppression des lignes ,Remplacement des valeurs manquantes(NULL,mode..))

In [21]:
# suppression des lignes sans Numéro de comptes (dropna)
before = len(df)
df = df.dropna(subset=["ACCOUNT_NO"])
print(f"\n[ACCOUNT_NO]  9.1% nulls")
print(f"  ✓ Lignes sans compte supprimées : {before - len(df):,}  → {len(df):,} restantes")


[ACCOUNT_NO]  9.1% nulls
  ✓ Lignes sans compte supprimées : 0  → 445,803 restantes


##### Statut martial 

In [ ]:
# Remplacement des valeurs nulles par le Mode (M)
mode_marital = df["MARITAL_STATUS"].mode()[0]
df["MARITAL_STATUS"] = df["MARITAL_STATUS"].fillna(mode_marital)
print(f"\n[MARITAL_STATUS]  17.8% nulls")
print(f"  ✓ Imputé par le mode : '{mode_marital}' (Marié — 49% du volume)")


[MARITAL_STATUS]  17.8% nulls
  ✓ Imputé par le mode : 'M' (Marié — 49% du volume)


##### Score KYC

In [9]:
# Remplacement des valeurs nulles par le mode (LR)
mode_kyc = df["SCORE_KYC"].mode()[0]
df["SCORE_KYC"] = df["SCORE_KYC"].fillna(mode_kyc)
print(f"\n[SCORE_KYC]  0.2% nulls")
print(f"  ✓ Imputé par le mode : '{mode_kyc}' (Low Risk — 69% du volume)")


[SCORE_KYC]  0.2% nulls
  ✓ Imputé par le mode : 'LR' (Low Risk — 69% du volume)


##### Completed file 

In [11]:
# Remplacement des valeurs nulles par (NO) : logique métier 
df["COMPLETED_FILE"] = df["COMPLETED_FILE"].fillna("NO")
print(f"\n[COMPLETED_FILE]  48% nulls")
print(f"  ✓ Null = dossier non complété → imputé par 'NO' (logique métier)")


[COMPLETED_FILE]  48% nulls
  ✓ Null = dossier non complété → imputé par 'NO' (logique métier)


##### Currency

In [13]:
# Remplacement des valeurs nulles par le mode : TND
mode_currency = df["CURRENCY"].mode()[0]
df["CURRENCY"] = df["CURRENCY"].fillna(mode_currency)
print(f"\n[CURRENCY]  29.6% nulls")
print(f"  ✓ Imputé par '{mode_currency}' (Dinar Tunisien — 96% des devises renseignées)")


[CURRENCY]  29.6% nulls
  ✓ Imputé par 'TND' (Dinar Tunisien — 96% des devises renseignées)


##### Product & Account_nature

In [14]:
cols = ["ACCOUNTNATURE", "PRODUCT"]
df[cols] = df[cols].fillna("NULL")
print(df[["ACCOUNTNATURE", "PRODUCT"]].head())

                       ACCOUNTNATURE                        PRODUCT
0  Crédit acquisition logement TEGF6       RT.RT.CRD.IMMOBILERS.527
1                     DEPOTS A TERME               BANK.CAT.NEG.SIM
2                  Crédit rénovation       RT.RT.CRD.IMMOBILERS.548
3  Compte Allocation Touristique TND  BANK.CUR.ACCT.ALL.TOURS.CARTE
4                     DEPOTS A TERME                           NULL


##### CLOSURE REASON

In [23]:
# Changement de la cause de fermeture : si le compte est actif par "non fermé", si le compte est 'closed' par "Inconnue"
df.loc[df["ACCOUNT_STATUS"] == "Active", "CLOSURE_REASON"] = \
    df.loc[df["ACCOUNT_STATUS"] == "Active", "CLOSURE_REASON"].fillna("Non fermé")
df.loc[df["ACCOUNT_STATUS"] == "Closed", "CLOSURE_REASON"] = \
    df.loc[df["ACCOUNT_STATUS"] == "Closed", "CLOSURE_REASON"].fillna("INCONNUE")
print(f"\n[CLOSURE_REASON]  71.5% nulls")
print(f"  ✓ Comptes actifs  → 'NON_FERME'  (logique métier : pas encore fermé)")
print(f"  ✓ Comptes fermés  → 'INCONNUE'   (raison non renseignée dans la source)")


[CLOSURE_REASON]  71.5% nulls
  ✓ Comptes actifs  → 'NON_FERME'  (logique métier : pas encore fermé)
  ✓ Comptes fermés  → 'INCONNUE'   (raison non renseignée dans la source)


### Traitement des valeurs manquantes quantitatives (Suppression des lignes ,Remplacement des valeurs manquantes(NULL,moyenne..))

#### transformation format des dates

In [28]:
# Conversion des colonnes de dates (stockées comme float YYYYMMDD)
date_cols_yyyymmdd = [
    "CUST_OPENING_DATE", "LAST_REVIEW_DATE", "NEXT__REVIEW_DATE",
    "ACCT_OPENING_DATE", "ACCT_CLOSE_DATE", "STARTDATE"
]
for col in date_cols_yyyymmdd:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col].dropna().astype(int).astype(str).reindex(df.index),
            format="%Y%m%d", errors="coerce"
        )

#Conversion des Date of birth vers le format date
df["DATE_OF_BIRTH"] = pd.to_numeric(df["DATE_OF_BIRTH"], errors="coerce")
df["DATE_OF_BIRTH"] = pd.to_datetime(
    df["DATE_OF_BIRTH"].dropna().astype(int).astype(str).reindex(df.index),
    format="%Y", errors="coerce"
)

##### Cust_openning_date

In [ ]:
# distribution casi symétrique (médiane ou moyenne , médiane plus robuste)
median_cust = df["CUST_OPENING_DATE"].median()
df["CUST_OPENING_DATE"] = df["CUST_OPENING_DATE"].fillna(median_cust)
print(f"\n[CUST_OPENING_DATE]  2.8% nulls  |  skew ≈ 0")
print(f"  ✓ Imputé par la médiane : {median_cust.date()}")


[CUST_OPENING_DATE]  2.8% nulls  |  skew ≈ 0
  ✓ Imputé par la médiane : 2014-09-04


##### Date of birth

In [29]:
# Valeurs abérrantes donc imputation par la médiane (médiane seulement)
median_dob = df["DATE_OF_BIRTH"].median()
df["DATE_OF_BIRTH"] = df["DATE_OF_BIRTH"].fillna(median_dob)
print(f"\n[DATE_OF_BIRTH]  14.3% nulls  |  skew = -2.43 (asymétrie gauche)")
print(f"  ✓ Imputé par la médiane : {int(median_dob.year)}")


[DATE_OF_BIRTH]  14.3% nulls  |  skew = -2.43 (asymétrie gauche)
  ✓ Imputé par la médiane : 1978


##### last review date & Next review

In [32]:
median_last = df["LAST_REVIEW_DATE"].median()
df["LAST_REVIEW_DATE"] = df["LAST_REVIEW_DATE"].fillna(median_last)
print(f"\n[LAST_REVIEW_DATE]  5.6% nulls  |  skew = -1.70")
print(f"  ✓ Imputé par la médiane : {median_last.date()}")

median_next = df["NEXT__REVIEW_DATE"].median()
df["NEXT__REVIEW_DATE"] = df["NEXT__REVIEW_DATE"].fillna(median_last)
print(f"\n[NEXT__REVIEW_DATE]  5.0% nulls  |  skew = -1.81")
print(f"  ✓ Imputé par la médiane : {median_next.date()}")


[LAST_REVIEW_DATE]  5.6% nulls  |  skew = -1.70
  ✓ Imputé par la médiane : NaT

[NEXT__REVIEW_DATE]  5.0% nulls  |  skew = -1.81
  ✓ Imputé par la médiane : NaT


##### Starting date

In [33]:
df["STARTDATE"] = pd.to_datetime(df["STARTDATE"], errors="coerce")
print(f"\n[STARTDATE]  86.8% nulls")
print(f"  ✓ Laissé NaT — uniquement LENDING & DEPOSITS, absent sur les autres produits")


[STARTDATE]  86.8% nulls
  ✓ Laissé NaT — uniquement LENDING & DEPOSITS, absent sur les autres produits


#### Account_opening_date et account_category avec des valeurs null ont été supprimées 

##### Account_balance

In [35]:
df["ACCT_BALANCE"] = pd.to_numeric(df["ACCT_BALANCE"], errors="coerce")
print(f"\n[ACCT_BALANCE]  29.6% nulls  |  skew = 21.67 (très asymétrique)")
print(f"  ✓ Laissé NaN — distribution trop hétérogène pour une imputation fiable")


[ACCT_BALANCE]  29.6% nulls  |  skew = 21.67 (très asymétrique)
  ✓ Laissé NaN — distribution trop hétérogène pour une imputation fiable
